In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
import optuna

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

In [6]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from Models.mtcnn import *

In [ ]:
inputs = np.load("../Data/multitask_data/inputs.npy")
input_shape = inputs.shape[1:]
print(input_shape)
output_dir = "../Data/multitask_data/output/"
output_list = [f for f in os.listdir(output_dir)]
output_list = output_list[0:1]
print(output_list)

dataset_candidates = ["real_data/dataset.csv", "./data/dataset.csv", "../Data/unitask_data/dataset.csv"]
dataset_path = next((path for path in dataset_candidates if os.path.exists(path)), None)
if dataset_path is None:
    raise FileNotFoundError(f"Could not find the CV dataset. Tried: {dataset_candidates}")
points = pd.read_csv(dataset_path).to_numpy(dtype=float)
if len(points) != len(inputs):
    raise ValueError(f"CV dataset has {len(points)} rows but inputs has {len(inputs)} samples")
n_covariates = inputs.shape[-1]
region ='all'

folds_by_method = build_folds_by_method(
    region, points=points, env=points[:, :n_covariates], tar=points[:, n_covariates:n_covariates + 7],
    n_folds=5, block_sp_threshold=[10000, 20000, 30000, 50000], sp1_threshold=[10000, 20000, 30000, 50000],
    sp_threshold=[10000, 20000, 30000, 50000],
    random_repeats=1, random_state=42,
)

def folds_to_splits(folds, n_samples):
    fold_map = {int(row[0]): int(row[1]) for row in folds}
    if set(fold_map) != set(range(n_samples)):
        raise ValueError("CV fold assignments do not cover every input sample exactly")
    splits = []
    for fold_id in sorted(set(fold_map.values())):
        val_idx = np.array([idx for idx, value in fold_map.items() if value == fold_id], dtype=int)
        splits.append((np.setdiff1d(np.arange(n_samples), val_idx), val_idx))
    return splits
cv_splits_by_method = {m: folds_to_splits(folds_by_method[m], len(inputs)) for m in folds_by_method.keys()}


(7, 7, 38)
['outputA%.npy']
d:\DSM_MODULE\CVs\cv_folds\all\random
d:\DSM_MODULE\CVs\cv_folds\all\sp1
d:\DSM_MODULE\CVs\cv_folds\all\sp2
d:\DSM_MODULE\CVs\cv_folds\all\sp
Building folds by method in d:\DSM_MODULE\CVs\cv_folds\all...
  - Random CV: 1 repeats, k=5
Generating random-CV folds.
Checking for existing CSV: d:\DSM_MODULE\CVs\cv_folds\all\random\randomCV_rep0_k5.csv
  - Block CV: k=5, thresholds=[10000, 20000, 30000, 50000]
    - Threshold: 10000
Generating block-CV folds.
    - Threshold: 20000
Generating block-CV folds.
    - Threshold: 30000
Generating block-CV folds.
    - Threshold: 50000
Generating block-CV folds.
  - SP1 CV: k=5, thresholds=[10000, 20000, 30000, 50000]
    - Threshold: 10000
    - Threshold: 20000
    - Threshold: 30000
    - Threshold: 50000
  - SP2 CV: k=5
  - SP CV: k=5, thresholds=[10000, 20000, 30000, 50000]
    - Threshold: 10000
    - Threshold: 20000
    - Threshold: 30000
    - Threshold: 50000


In [12]:
def objectiveCV(trial, inputs, y, input_shape, cv_splits):
    K.clear_session()
    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "adamw"])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    scaler_type = trial.suggest_categorical("scaler", ["standard", "minmax"])
    num_tasks = y.shape[1] if len(y.shape) > 1 and y.shape[1] > 1 else 1
    fold_val_losses = []
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        callbacks = [TFKerasPruningCallback(trial, "val_loss"), EarlyStopping(monitor="val_loss", patience=10)]
        x_train, x_val, y_train, y_val = inputs[train_idx], inputs[val_idx], y[train_idx], y[val_idx]
        scaler = MinMaxScaler() if scaler_type == "minmax" else StandardScaler()
        x_train = scaler.fit_transform(x_train.reshape(-1, x_train.shape[-1])).reshape(x_train.shape)
        x_val = scaler.transform(x_val.reshape(-1, x_val.shape[-1])).reshape(x_val.shape)
        optimizer = Adam(learning_rate=learning_rate) if optimizer_name == "adam" else AdamW(learning_rate=learning_rate, weight_decay=1e-4)
        model = create_cnn_optuna(trial, input_shape)
        model.compile(optimizer=optimizer, loss=["mse"] * num_tasks)
        y_train_fit = y_train if num_tasks == 1 else [y_train[:, i] for i in range(num_tasks)]
        y_val_fit = y_val if num_tasks == 1 else [y_val[:, i] for i in range(num_tasks)]
        history = model.fit(x_train, y_train_fit, validation_data=(x_val, y_val_fit), batch_size=batch_size, epochs=50, callbacks=callbacks, verbose=0)
        fold_best = float(min(history.history["val_loss"]))
        fold_val_losses.append(fold_best)
        trial.report(float(np.mean(fold_val_losses)), step=fold)
        if trial.should_prune(): raise optuna.TrialPruned()
        del model
        K.clear_session()
    final_score = float(np.mean(fold_val_losses))
    print(f"Trial {trial.number} finished with mean val_loss = {final_score}")
    return final_score


In [ ]:
hyperpramateres_tunning = False
CV_METHODS = [ 'block_k5_20000', 'block_k5_30000', 'block_k5_50000',   'block_k5_10000', 
              'sp1_k5_20000', 'sp1_k5_30000', 'sp1_k5_50000',   'sp1_k5_10000',
               'sp2_k5', 'sp_k5_20000', 'sp_k5_30000', 'sp_k5_50000',   'sp_k5_10000', ]

if hyperpramateres_tunning:
    list_studies = []
    for output in output_list:
        y = np.load(os.path.join(output_dir, output))
        output_name = output.split(".")[0]
        for cv_method in CV_METHODS:
            save_path = os.path.join("./cv_optuna", cv_method, output_name)
            os.makedirs(save_path, exist_ok=True)
            print(f"Currently working on --> {output_name} ({cv_method})")
            study = optuna.create_study(direction="minimize")
            study.optimize(lambda trial, splits=cv_splits_by_method[cv_method]: objectiveCV(trial, inputs, y, input_shape, splits), n_trials=60, show_progress_bar=False, n_jobs=1)
            list_studies.append(study)
            pd.DataFrame([{"output_name": output, "cv_method": cv_method, "best_score": study.best_value, "best_params": study.best_params}]).to_csv(os.path.join(save_path, "cv_optuna_results.csv"), index=False)


# Model Evaluation By CV method 

In [14]:
from Models.cnn_cv_metrics import evaluate_multitask_cnn_cv
import ast
 
# Example:
# study is the Optuna study for one property and one CV method.
# For example, this can be study.best_params immediately after study.optimize(...)

CV_MEHODS = [ 'block_k5_20000', 'block_k5_30000', 'block_k5_50000',   'block_k5_10000', 
              'sp1_k5_20000', 'sp1_k5_30000', 'sp1_k5_50000',   'sp1_k5_10000',
               'sp2_k5', 'sp_k5_20000', 'sp_k5_30000', 'sp_k5_50000',   'sp_k5_10000', ]

CV_MEHODS = ['random']
output_dir = "../Data/multitask_data/output/"
output_list = [f for f in os.listdir(output_dir)]
cv_splits_by_method = {m: folds_to_splits(folds_by_method[m], len(inputs)) for m in CV_MEHODS}

print(output_list)

for cv_method in CV_MEHODS:
    for output in output_list[2:]:
        print('Output : ', output, 'cv_method:' , cv_method) 
        output_name = output.split(".")[0]
        print('Output name : ', output_name, 'cv_method:' , cv_method)
        best_params =  pd.read_csv(f"../Training/cv_optuna_mtcnn/random/{output_name}/kfold_optuna_results.csv").iloc[0]["best_params"]
        best_params = ast.literal_eval(best_params)
        print(best_params)
        y = np.load(os.path.join(output_dir, output))

        metrics_by_fold, overall_by_task, overall_metrics, y_true_oof, y_pred_oof,  = evaluate_multitask_cnn_cv(
            model_builder=create_cnn_optuna,
            best_params=best_params,
            inputs=inputs,
            y=y,
            input_shape=input_shape,
            cv_splits=cv_splits_by_method[cv_method],
            epochs=100,
            patience=10,
            seed=42,
            task_names=["0-5cm", "5-15cm", "15-30cm", "30-60cm", "60-100cm", "100-200cm"]
        )

        print("Metrics by fold:")
        display(metrics_by_fold)

        print("Overall metrics:")
        display(overall_metrics)

        # Save results
        print('end=> Saving results for output:', output, 'cv_method:' , cv_method)

        save_path = os.path.join("cv_optuna_mtcnn/test_best_random_on_spCV", cv_method, output_name)
        print('saving in path : ', save_path)
        os.makedirs(save_path, exist_ok=True)

        overall_by_task.to_csv(
            os.path.join(save_path, "metrics_by_task.csv"),
            index=False,
        )
        metrics_by_fold.to_csv(
            os.path.join(save_path, "metrics_by_fold.csv"),
            index=False,
        )

        overall_metrics.to_csv(
            os.path.join(save_path, "metrics_overall.csv"),
            index=False,
        )

        np.save(
        os.path.join(save_path, "y_true_oof.npy"),
        y_true_oof,
        )

        np.save(
            os.path.join(save_path, "y_pred_oof.npy"),
            y_pred_oof,
        )    



['outputA%.npy', 'outputCE.npy', 'outputCEC.npy', 'outputL%.npy', 'outputOM(%).npy', 'outputpH.npy', 'outputS%.npy']
Output :  outputCEC.npy cv_method: random
Output name :  outputCEC cv_method: random
{'optimizer': 'adamw', 'learning_rate': 0.00031867591942926605, 'batch_size': 32, 'scaler': 'standard', 'n_blocks': 1, 'block1_type': 'conv', 'filters_block1': 64, 'kernel_block1': 5, 'activation_block1': 'relu', 'dilation_block1': (1, 1), 'batchnorm_block1': True, 'cbam_block1': True, 'dropout_block1': 0.03420346230393939, 'pooling_block1': False, 'n_shared': 2, 'shared_units_1': 300, 'shared_activation_1': 'relu', 'shared_dropout_1': 0.1256159555853893, 'shared_units_2': 100, 'shared_activation_2': 'selu', 'shared_dropout_2': 0.2586515346089261, 'task_0_layers': 2, 'task_0_units_0': 128, 'task_0_activation_0': 'relu', 'task_0_dropout_0': 0.3311551434501271, 'task_0_units_1': 128, 'task_0_activation_1': 'selu', 'task_0_dropout_1': 0.1960502177442618, 'task_1_layers': 3, 'task_1_units_0'

,fold,target,MSE,RMSE,R2
0,1,0-5cm,6.932515,2.632967,0.426273
1,1,5-15cm,6.766592,2.601267,0.423127
2,1,15-30cm,9.960670,3.156053,0.314650
3,1,30-60cm,6.810055,2.609608,0.376891
4,1,60-100cm,8.015397,2.831148,0.719308
5,1,100-200cm,8.235125,2.869691,0.711614
6,2,0-5cm,20.081841,4.481277,0.204821
7,2,5-15cm,17.158532,4.142286,0.192373
8,2,15-30cm,16.853082,4.105250,0.170350
9,2,30-60cm,18.044240,4.247851,0.122908


Overall metrics:


,fold,target,MSE,RMSE,R2
0,overall,all_tasks,10.481502,3.237515,0.50344


end=> Saving results for output: outputCEC.npy cv_method: random
saving in path :  cv_optuna_mtcnn/test_best_random_on_spCV\random\outputCEC
Output :  outputL%.npy cv_method: random
Output name :  outputL% cv_method: random
{'optimizer': 'adamw', 'learning_rate': 0.00036252409935045687, 'batch_size': 8, 'scaler': 'standard', 'n_blocks': 1, 'block1_type': 'conv', 'filters_block1': 128, 'kernel_block1': 5, 'activation_block1': 'swish', 'dilation_block1': (1, 1), 'batchnorm_block1': False, 'cbam_block1': True, 'dropout_block1': 0.1554748925991262, 'pooling_block1': False, 'n_shared': 1, 'shared_units_1': 100, 'shared_activation_1': 'relu', 'shared_dropout_1': 0.02063210900312, 'task_0_layers': 1, 'task_0_units_0': 128, 'task_0_activation_0': 'relu', 'task_0_dropout_0': 0.198298030966146, 'task_1_layers': 1, 'task_1_units_0': 32, 'task_1_activation_0': 'elu', 'task_1_dropout_0': 0.20418338698846794, 'task_2_layers': 2, 'task_2_units_0': 64, 'task_2_activation_0': 'selu', 'task_2_dropout_0'

,fold,target,MSE,RMSE,R2
0,1,0-5cm,104.966124,10.245298,0.711985
1,1,5-15cm,81.477762,9.026503,0.758150
2,1,15-30cm,80.041542,8.946594,0.765209
3,1,30-60cm,44.717127,6.687087,0.861219
4,1,60-100cm,42.301271,6.503943,0.877961
5,1,100-200cm,45.083917,6.714456,0.869933
6,2,0-5cm,85.121850,9.226150,0.770344
7,2,5-15cm,80.072471,8.948322,0.777164
8,2,15-30cm,69.802729,8.354803,0.791675
9,2,30-60cm,69.478502,8.335377,0.774588


... (11 outputs left out)

,fold,target,MSE,RMSE,R2
0,1,0-5cm,149.169246,12.213486,0.783265
1,1,5-15cm,128.232550,11.323981,0.807670
2,1,15-30cm,121.980824,11.044493,0.809220
3,1,30-60cm,88.830566,9.424997,0.858600
4,1,60-100cm,75.696129,8.700352,0.902715
5,1,100-200cm,73.187961,8.554996,0.905939
6,2,0-5cm,119.106150,10.913576,0.816137
7,2,5-15cm,126.362546,11.241110,0.803351
8,2,15-30cm,109.967632,10.486545,0.818915
9,2,30-60cm,96.257025,9.811066,0.835518


Overall metrics:


,fold,target,MSE,RMSE,R2
0,overall,all_tasks,104.164709,10.206111,0.849053


end=> Saving results for output: outputS%.npy cv_method: random
saving in path :  cv_optuna_mtcnn/test_best_random_on_spCV\random\outputS%
